In [7]:
import numpy as np
import math
from sklearn.model_selection import train_test_split
import torch.nn as nn
import matplotlib.pyplot as plt
import random
import matplotlib as mpl
import os
import gc
import pandas as pd
import csv
from numpy import *
from datetime import date
import time
import builtins
from sax import sax_tokenizer

In [8]:
x_mfcc = np.load('./x_mfcc.npy', allow_pickle=True)
x_raw = np.zeros(x_mfcc.shape)
y = np.load('./y.npy', allow_pickle=True)

x_raw = x_raw.transpose(0,2,1)
x_mfcc = x_mfcc.transpose(0,2,1)

In [9]:
# Assuming x_raw, x_mfcc, and y are already defined numpy arrays

# First split: Train (70%) and Temp (30%)

## seeds = [42, 117]
seed = 42
x_raw_train, x_raw_temp, x_mfcc_train, x_mfcc_temp, y_train, y_temp = train_test_split(
    x_raw, x_mfcc, y, test_size=0.30, random_state=seed, stratify=y
)

# Second split: Validation (15%) and Test (15%) from Temp (30%)
x_raw_val, x_raw_test, x_mfcc_val, x_mfcc_test, y_val, y_test = train_test_split(
    x_raw_temp, x_mfcc_temp, y_temp, test_size=0.50, random_state=seed, stratify=y_temp
)


x_raw_val = np.concatenate([x_raw_val, x_raw_test], axis=0)
x_mfcc_val = np.concatenate([x_mfcc_val, x_mfcc_test], axis=0)
y_val = np.concatenate([y_val, y_test], axis=0)

x_raw_test = x_raw_val
x_mfcc_test = x_mfcc_val
y_test = y_val

print(x_raw_train.shape, x_mfcc_train.shape, y_train.shape)
print(x_raw_val.shape, x_mfcc_val.shape, y_val.shape)
print(x_raw_test.shape, x_mfcc_test.shape, y_test.shape)



np.save('x_train.npy', x_mfcc_train)
np.save('x_valid.npy', x_mfcc_val)
np.save('x_test.npy', x_mfcc_test)

np.save('y_train.npy', y_train)
np.save('y_valid.npy', y_val)
np.save('y_test.npy', y_test)

(280, 501, 40) (280, 501, 40) (280,)
(120, 501, 40) (120, 501, 40) (120,)
(120, 501, 40) (120, 501, 40) (120,)


In [10]:
category = 10
word_len = 1

def convert_to_sax(input_x):
    x_sax = np.zeros(input_x.shape)
    for i in range(len(x_sax)):
        start = 0
        for j in range(input_x.shape[-1]):
            temp = sax_tokenizer(input_x[i,:,j].tolist(),alphabet_size=category, word_length=word_len) #+ start
            x_sax[i,:,j] =  np.array(temp) + start
            start += category
        if i%100 == 0:
            print(f'Done with {i}')        
    return x_sax      

In [11]:
idx = 1
x_train = np.load(f'./x_train.npy', allow_pickle=True)
x_valid = np.load(f'./x_valid.npy', allow_pickle=True)
x_test = np.load(f'./x_test.npy', allow_pickle=True)

# x_train = np.load(f'./x_raw_train_{idx}.npy', allow_pickle=True)
# x_valid = np.load(f'./x_raw_valid_{idx}.npy', allow_pickle=True)
# x_test = np.load(f'./x_raw_test_{idx}.npy', allow_pickle=True)

sax_train = convert_to_sax(x_train)
sax_valid = convert_to_sax(x_valid)
sax_test = convert_to_sax(x_test)

assert x_train.shape[0]==sax_train.shape[0]
assert x_valid.shape[0]==sax_valid.shape[0]
assert x_test.shape[0]==sax_test.shape[0]

Done with 0
Done with 100
Done with 200
Done with 0
Done with 100
Done with 0
Done with 100


In [12]:
print(sax_train.shape, sax_valid.shape, sax_test.shape)

(280, 501, 40) (120, 501, 40) (120, 501, 40)


In [13]:
np.unique(sax_train), len(np.unique(sax_train))

(array([  0.,   1.,   2.,   3.,   4.,   5.,   6.,   7.,   8.,   9.,  10.,
         11.,  12.,  13.,  14.,  15.,  16.,  17.,  18.,  19.,  20.,  21.,
         22.,  23.,  24.,  25.,  26.,  27.,  28.,  29.,  30.,  31.,  32.,
         33.,  34.,  35.,  36.,  37.,  38.,  39.,  40.,  41.,  42.,  43.,
         44.,  45.,  46.,  47.,  48.,  49.,  50.,  51.,  52.,  53.,  54.,
         55.,  56.,  57.,  58.,  59.,  60.,  61.,  62.,  63.,  64.,  65.,
         66.,  67.,  68.,  69.,  70.,  71.,  72.,  73.,  74.,  75.,  76.,
         77.,  78.,  79.,  80.,  81.,  82.,  83.,  84.,  85.,  86.,  87.,
         88.,  89.,  90.,  91.,  92.,  93.,  94.,  95.,  96.,  97.,  98.,
         99., 100., 101., 102., 103., 104., 105., 106., 107., 108., 109.,
        110., 111., 112., 113., 114., 115., 116., 117., 118., 119., 120.,
        121., 122., 123., 124., 125., 126., 127., 128., 129., 130., 131.,
        132., 133., 134., 135., 136., 137., 138., 139., 140., 141., 142.,
        143., 144., 145., 146., 147., 

In [14]:
def onehotencoding(x_sax):
    x_sax = x_sax.astype(int)
    x_sax_ohe = np.zeros((x_sax.shape[0], x_sax.shape[1], category*x_sax.shape[-1]))

    for i in range(len(x_sax_ohe)):
        for j in range(x_sax_ohe.shape[1]): 
            idx = x_sax[i,j,:].tolist()
            x_sax_ohe[i,j,idx] = 1
    
    return x_sax_ohe

In [15]:
sax_train_ohe = onehotencoding(sax_train)
sax_valid_ohe = onehotencoding(sax_valid)
sax_test_ohe = onehotencoding(sax_test)

In [16]:
print(sax_train[100,0:2,:], sax_train_ohe[100,1,:])
# print(sax_valid[100,0:10,:], sax_valid_ohe[100,2,:])

[[  6.  14.  23.  31.  45.  53.  67.  77.  86.  95. 107. 119. 124. 136.
  149. 156. 162. 175. 186. 195. 203. 210. 220. 232. 240. 255. 266. 271.
  283. 290. 308. 313. 329. 338. 344. 357. 364. 377. 380. 396.]
 [  7.  18.  22.  32.  48.  57.  68.  79.  87.  95. 109. 117. 123. 136.
  146. 150. 160. 177. 187. 193. 200. 210. 221. 233. 243. 253. 266. 278.
  284. 297. 308. 317. 323. 331. 342. 350. 364. 373. 380. 399.]] [0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 1. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 1. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0.
 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 1. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 0. 0. 0. 0.
 

In [17]:
np.save('./sax_train', sax_train_ohe)
np.save('./sax_valid', sax_valid_ohe)
np.save('./sax_test', sax_test_ohe)